# 04 ― JMA AMeDAS の最新観測

気象庁 AMeDAS の 10 分値スナップショットを取得して、最寄り観測点を抽出する。

ポイント:

- `JmaAmedasService` はディスクキャッシュ込みのフェッチャ (連続呼び出しに耐える)
- `nearest_stations()` で指定緯度経度から haversine 距離順に並べる
- 観測値は `AmedasSnapshot.observations[station_id]` に `dict[str, float]` で入る

**データ帰属**: 出典 気象庁ホームページ
(複数地点の値を合成・加工した図には「処理データ」と明記すること)


In [ ]:
from aiseed_weather.models.user_settings import load_or_init, resolved_data_dir
from aiseed_weather.services.jma_amedas_service import (
    JmaAmedasService, nearest_stations,
)

data_dir = resolved_data_dir(load_or_init().settings)
service = JmaAmedasService(data_dir=data_dir)


## 最新スナップショットを取得


In [ ]:
snapshot = await service.fetch()
print(f"観測時刻      : {snapshot.timestamp}")
print(f"取得時刻      : {snapshot.fetched_at}")
print(f"観測点数      : {len(snapshot.observations)}")

# 1 地点の中身を覗く
first_id = next(iter(snapshot.observations))
print(f"\n観測点 {first_id} の観測値:")
for k, v in snapshot.observations[first_id].items():
    print(f"  {k}: {v}")


## 観測点テーブル (位置・名前) を取得


In [ ]:
stations = await service.stations()
print(f"AMeDAS 観測点総数: {len(stations)}")

# 適当な観測点を 5 つ眺める
for sid in list(stations.keys())[:5]:
    s = stations[sid]
    print(f"  {sid}  {s.name_kanji:8s}  ({s.latitude:.3f}, {s.longitude:.3f})  type={s.station_type}")


## 徳島近傍の最寄り観測点 5 つ


In [ ]:
nearby = nearest_stations(
    stations, latitude=33.78, longitude=134.49, limit=5,
    station_types=("A",),  # 'A' = 気温・降水・風など完備
)
for sid, distance_km in nearby:
    s = stations[sid]
    obs = snapshot.observations.get(sid, {})
    temp = obs.get("temp")  # 気温 (°C)
    print(f"  {s.name_kanji:8s}  {distance_km:5.1f} km  気温={temp}°C")


## 複数地点比較 ― 大都市の現況気温


In [ ]:
import polars as pl

CITIES = {
    "札幌":   (43.07, 141.35),
    "仙台":   (38.27, 140.87),
    "東京":   (35.69, 139.69),
    "名古屋": (35.18, 136.91),
    "大阪":   (34.69, 135.50),
    "福岡":   (33.59, 130.40),
    "徳島":   (33.78, 134.49),
    "那覇":   (26.21, 127.68),
}

rows: list[dict] = []
for city, (lat, lon) in CITIES.items():
    [(sid, dist)] = nearest_stations(
        stations, latitude=lat, longitude=lon, limit=1,
        station_types=("A",),
    )
    s = stations[sid]
    obs = snapshot.observations.get(sid, {})
    rows.append({
        "city":     city,
        "station":  s.name_kanji,
        "dist_km":  round(dist, 1),
        "temp_C":   obs.get("temp"),
        "wind_mps": obs.get("wind"),
        "precip1h_mm": obs.get("precipitation1h"),
    })

pl.DataFrame(rows)


## 自由解析のヒント

- `service.fetch(force=True)` で 10 分キャッシュを破棄して再取得
- `obs` キーは `temp` (気温) / `wind` (風速) / `windDirection` /
  `precipitation1h` (前 1h 降水) / `precipitation10m` / `humidity` /
  `pressure` (海面更正気圧) など。観測点タイプ ('A'/'B'/'C') で揃っている項目が違う
- `station_types=("A", "B")` で型を絞り込める ('C' は降水量のみの観測点)
- レーダー (5 分更新の雨雲) は `JmaRadarService` を使う ― 構成は AMeDAS と同様

**注意**: 観測値を別地点と合成したり、補間したり、加工して図にする場合、
気象庁の規約で「処理データ」明記が必要。

**参考**: AMeDAS のフィールド名一覧は気象庁の公開仕様参照
